# ArSL Graduation Model — Live Webcam Test (131 classes)

Real-time test for **`arsl_custom_best.h5`** (100 words + 31 numbers).

| Setting | Value |
|---------|-------|
| Sequence | **48 frames** (matches training) |
| Features | **258** (Holistic: pose + both hands) |
| MediaPipe | `model_complexity=0` |

**Controls:** `Q` quit · `R` reset sentence · `SPACE` add space · `BACKSPACE` delete last word · `C` clear buffer

Run cells **1 → 5** in order. Cell 4 verifies offline inference before opening the camera.

In [1]:
# Cell 1: Imports & GPU
import time
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import tensorflow as tf
from pathlib import Path
from collections import deque

print(f'TensorFlow {tf.__version__} | OpenCV {cv2.__version__} | MediaPipe {mp.__version__}')

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU: {gpus[0].name} (memory growth ON)')
else:
    print('CPU inference')

# Arabic overlay (optional)
try:
    from PIL import Image, ImageDraw, ImageFont
    import arabic_reshaper
    from bidi.algorithm import get_display
    _AR_FONT = 'C:/Windows/Fonts/arial.ttf'
    _font_cache = {}

    def draw_arabic(frame, text, pos, size=28, color=(255, 255, 255)):
        try:
            t = get_display(arabic_reshaper.reshape(str(text)))
        except Exception:
            t = str(text)
        if size not in _font_cache:
            try:
                _font_cache[size] = ImageFont.truetype(_AR_FONT, size)
            except Exception:
                _font_cache[size] = ImageFont.load_default()
        pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ImageDraw.Draw(pil).text(pos, t, font=_font_cache[size], fill=(color[2], color[1], color[0]))
        return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
    HAS_ARABIC = True
except ImportError:
    HAS_ARABIC = False
    def draw_arabic(frame, text, pos, size=28, color=(255, 255, 255)):
        cv2.putText(frame, str(text), pos, cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        return frame
    print('Tip: pip install arabic-reshaper python-bidi pillow for Arabic HUD text')


TensorFlow 2.10.0 | OpenCV 4.11.0 | MediaPipe 0.10.9
GPU: /physical_device:GPU:0 (memory growth ON)


In [2]:
# Cell 2: Graduation model paths (131-class)
DIR = Path(r'M:/Term 10/Grad/SLR Main/Words/ArSL Word (Arabic)')

MODEL_PATH  = DIR / 'arsl_custom_best.h5'
SCALER_PATH = DIR / 'arsl_custom_scaler.npz'
CLASSES_CSV = DIR / 'arsl_custom_classes.csv'
SUBSET_NPZ  = DIR / 'arsl_custom_subset.npz'  # offline smoke test only

SEQUENCE_LENGTH = 48
NUM_FEATURES    = 258
EXPECTED_CLASSES = 131

# Live tuning — 131 classes need slightly stricter gates than small vocab
CONFIDENCE_THRESHOLD = 0.42
PREDICTION_INTERVAL    = 0.55   # seconds between model calls
STABILITY_WINDOW       = 4      # same label N times → confirm word
COOLDOWN_TIME          = 1.8    # seconds after confirm before next
MIN_ACTIVE_FRAMES      = 0.35   # fraction of non-zero frames in buffer

CAMERA_INDEX   = 0
CAPTURE_W, CAPTURE_H = 640, 480   # fast capture
DISPLAY_W, DISPLAY_H = 960, 540   # lighter than 1280×720
DRAW_LANDMARKS = True
SHOW_TOP_K     = 5

for p, label in [(MODEL_PATH, 'model'), (SCALER_PATH, 'scaler'), (CLASSES_CSV, 'classes')]:
    if not p.exists():
        raise FileNotFoundError(f'Missing {label}: {p}')
print('Artifacts OK')
print(f'  Model   : {MODEL_PATH.name}')
print(f'  Seq×feat: {SEQUENCE_LENGTH}×{NUM_FEATURES} | classes: {EXPECTED_CLASSES}')


Artifacts OK
  Model   : arsl_custom_best.h5
  Seq×feat: 48×258 | classes: 131


In [3]:
# Cell 3: Load model, scaler, vocabulary + offline smoke test
print('Loading graduation model...')
model = tf.keras.models.load_model(str(MODEL_PATH), compile=False)
in_shape, out_shape = model.input_shape, model.output_shape
assert in_shape[1:] == (SEQUENCE_LENGTH, NUM_FEATURES), f'Bad input {in_shape}'
assert out_shape[-1] == EXPECTED_CLASSES, f'Bad output {out_shape}'
print(f'  Params: {model.count_params():,} | in={in_shape} out={out_shape[-1]} classes')

sc = np.load(str(SCALER_PATH))
scaler_mean  = sc['mean'].astype(np.float32)
scaler_scale = sc['scale'].astype(np.float32)
assert scaler_mean.shape == (NUM_FEATURES,)
print(f'  Scaler loaded ({NUM_FEATURES}-dim)')

class_df = pd.read_csv(CLASSES_CSV)
required = {'model_class_index', 'karsl_class_id', 'english', 'arabic'}
if not required.issubset(class_df.columns):
    raise ValueError(f'Classes CSV needs {required}, got {list(class_df.columns)}')

index_to_en = dict(zip(class_df['model_class_index'].astype(int), class_df['english'].astype(str)))
index_to_ar = dict(zip(class_df['model_class_index'].astype(int), class_df['arabic'].astype(str)))
print(f'  Vocabulary: {len(index_to_en)} labels')

# Warm-up + offline check (one held-out style sample from subset NPZ)
if SUBSET_NPZ.exists():
    with np.load(str(SUBSET_NPZ), mmap_mode='r') as d:
        idx = 100
        x = np.array(d['X'][idx:idx+1], dtype=np.float32)
        true_cid = int(d['y'][idx])
    x = (x - scaler_mean) / scaler_scale
    probs = model(x, training=False).numpy()[0]
    pred = int(np.argmax(probs))
    cid_to_idx = dict(zip(class_df['karsl_class_id'], class_df['model_class_index']))
    true_idx = int(cid_to_idx.get(true_cid, -1))
    ok = pred == true_idx
    print(f'  Offline smoke #{idx}: pred={index_to_en.get(pred,"?")} true={index_to_en.get(true_idx,"?")} {"OK" if ok else "MISMATCH"} ({probs[pred]:.1%})')
else:
    _ = model(np.zeros((1, SEQUENCE_LENGTH, NUM_FEATURES), np.float32), training=False)
    print('  Model warm-up done (no subset NPZ for smoke test)')

print('\nReady for live test — run Cell 4 (MediaPipe) then Cell 5 (webcam).')


Loading graduation model...
  Params: 1,642,819 | in=(None, 48, 258) out=131 classes
  Scaler loaded (258-dim)
  Vocabulary: 131 labels


MemoryError: Unable to allocate 313. MiB for an array with shape (81932544,) and data type float32

In [ ]:
# Cell 4: MediaPipe Holistic (must match training extraction)
mp_holistic = mp.solutions.holistic
mp_hands    = mp.solutions.hands
mp_draw     = mp.solutions.drawing_utils
mp_styles   = mp.solutions.drawing_styles

holistic = mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=0,
    enable_segmentation=False,
    refine_face_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

def extract_holistic_258(frame_bgr):
    """Return (258-dim vector, draw_items). Run on capture-size frame for speed."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    res = holistic.process(rgb)
    draw_items = []

    if res.pose_landmarks:
        pose = np.array([[lm.x, lm.y, lm.z, lm.visibility]
                         for lm in res.pose_landmarks.landmark], dtype=np.float32).flatten()
        draw_items.append(('pose', res.pose_landmarks))
    else:
        pose = np.zeros(132, dtype=np.float32)

    if res.left_hand_landmarks:
        lh = np.array([[lm.x, lm.y, lm.z] for lm in res.left_hand_landmarks.landmark],
                      dtype=np.float32).flatten()
        draw_items.append(('hand', res.left_hand_landmarks))
    else:
        lh = np.zeros(63, dtype=np.float32)

    if res.right_hand_landmarks:
        rh = np.array([[lm.x, lm.y, lm.z] for lm in res.right_hand_landmarks.landmark],
                      dtype=np.float32).flatten()
        draw_items.append(('hand', res.right_hand_landmarks))
    else:
        rh = np.zeros(63, dtype=np.float32)

    return np.concatenate([pose, lh, rh]), draw_items, bool(res.left_hand_landmarks or res.right_hand_landmarks)

print('MediaPipe Holistic ready (258-dim, complexity=0)')


In [ ]:
# Cell 5: Live webcam loop
FONT = cv2.FONT_HERSHEY_SIMPLEX
GREEN, RED, WHITE, BLACK = (0, 200, 0), (0, 0, 220), (255, 255, 255), (0, 0, 0)
YELLOW, CYAN, GRAY, LGRAY = (0, 210, 210), (255, 200, 0), (90, 90, 90), (180, 180, 180)

def run_live():
    cap = cv2.VideoCapture(CAMERA_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAPTURE_W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAPTURE_H)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    if not cap.isOpened():
        print('Cannot open webcam'); return

    print(f'Camera {CAPTURE_W}×{CAPTURE_H} → display {DISPLAY_W}×{DISPLAY_H}')
    print('Q=quit  R=reset  SPACE=space  BACKSPACE=delete  C=clear buffer')

    buf = deque(maxlen=SEQUENCE_LENGTH)
    stab = deque(maxlen=STABILITY_WINDOW)
    sent_en, sent_ar = [], []
    cur_en, cur_ar, cur_conf = '', '', 0.0
    topk = []
    last_pred = last_confirm = 0.0
    fps_q = deque(maxlen=20)
    seq_batch = np.zeros((1, SEQUENCE_LENGTH, NUM_FEATURES), dtype=np.float32)

    while True:
        t0 = time.time()
        ok, raw = cap.read()
        if not ok:
            break
        raw = cv2.flip(raw, 1)

        vec, draw_items, hands_on = extract_holistic_258(raw)
        buf.append(vec)

        frame = cv2.resize(raw, (DISPLAY_W, DISPLAY_H))
        h, w = frame.shape[:2]
        now = time.time()

        if DRAW_LANDMARKS:
            for kind, lm in draw_items:
                if kind == 'hand':
                    mp_draw.draw_landmarks(
                        frame, lm, mp_hands.HAND_CONNECTIONS,
                        mp_styles.get_default_hand_landmarks_style(),
                        mp_styles.get_default_hand_connections_style())
                elif kind == 'pose':
                    mp_draw.draw_landmarks(
                        frame, lm, mp_holistic.POSE_CONNECTIONS,
                        mp_styles.get_default_pose_landmarks_style())

        # Predict when buffer full
        if len(buf) == SEQUENCE_LENGTH and (now - last_pred) >= PREDICTION_INTERVAL:
            last_pred = now
            seq = np.array(buf, dtype=np.float32)
            active = np.mean(np.any(seq != 0, axis=1))
            if active >= MIN_ACTIVE_FRAMES:
                seq_batch[0] = (seq - scaler_mean) / scaler_scale
                proba = model(seq_batch, training=False).numpy()[0]
                pred_idx = int(np.argmax(proba))
                conf = float(proba[pred_idx])
                if conf >= CONFIDENCE_THRESHOLD:
                    cur_en = index_to_en.get(pred_idx, '?')
                    cur_ar = index_to_ar.get(pred_idx, '?')
                    cur_conf = conf
                    stab.append(cur_en)
                    if (len(stab) == STABILITY_WINDOW and len(set(stab)) == 1
                            and (now - last_confirm) >= COOLDOWN_TIME):
                        sent_en.append(cur_en)
                        sent_ar.append(cur_ar)
                        last_confirm = now
                        stab.clear()
                        print(f'  + "{cur_ar}" / {cur_en} ({cur_conf:.1%})')
                else:
                    cur_en = cur_ar = ''
                    cur_conf = 0.0
                top_idx = np.argsort(proba)[-SHOW_TOP_K:][::-1]
                topk = [(index_to_en.get(i,'?'), index_to_ar.get(i,'?'), float(proba[i])) for i in top_idx]
            else:
                cur_en = cur_ar = ''
                cur_conf = 0.0
                topk = []

        # HUD
        cv2.rectangle(frame, (0, 0), (w, 100), BLACK, -1)
        status = 'HANDS OK' if hands_on else 'NO HANDS'
        cv2.putText(frame, status, (10, 28), FONT, 0.7, GREEN if hands_on else RED, 2)
        cv2.putText(frame, f'Buffer {len(buf)}/{SEQUENCE_LENGTH}', (10, 58), FONT, 0.6, YELLOW, 2)
        if cur_en:
            cv2.putText(frame, f'{cur_en} ({cur_conf:.0%})', (10, 88), FONT, 0.65, CYAN, 2)
        for i, (en, ar, p) in enumerate(topk[:3]):
            cv2.putText(frame, f'{i+1}. {en} {p:.0%}', (w - 280, 28 + i * 24), FONT, 0.5, LGRAY, 1)

        sent_line_en = ' '.join(sent_en[-8:])
        sent_line_ar = ' '.join(sent_ar[-8:])
        cv2.rectangle(frame, (0, h - 70), (w, h), BLACK, -1)
        cv2.putText(frame, sent_line_en, (10, h - 42), FONT, 0.55, WHITE, 1)
        frame = draw_arabic(frame, sent_line_ar or '—', (10, h - 38), size=24)

        fps_q.append(1.0 / max(time.time() - t0, 1e-6))
        cv2.putText(frame, f'{np.mean(fps_q):.0f} FPS', (w - 90, h - 10), FONT, 0.5, GRAY, 1)

        cv2.imshow('ArSL Graduation Live (131 classes)', frame)
        key = cv2.waitKey(1) & 0xFF
        if key in (ord('q'), ord('Q')):
            break
        elif key in (ord('r'), ord('R')):
            sent_en.clear(); sent_ar.clear(); stab.clear()
            print('Sentence cleared')
        elif key == ord(' '):
            sent_en.append(' '); sent_ar.append(' ')
        elif key in (8, 127):  # backspace / delete
            if sent_en: sent_en.pop()
            if sent_ar: sent_ar.pop()
        elif key in (ord('c'), ord('C')):
            buf.clear(); stab.clear()
            print('Frame buffer cleared')

    cap.release()
    cv2.destroyAllWindows()
    holistic.close()
    print('\nFinal EN:', ' '.join(sent_en))
    print('Final AR:', ' '.join(sent_ar))

run_live()
